# 05 — Error analysis: correct vs. incorrect patients

This notebook is a first investigation into whether the baseline CNN is learning
useful coronary structures, before changing the architecture.

For one patient-level fold it pairs:

- **correctly classified patients**, and
- **incorrectly classified patients**

with, for each patient, a few sampled frames shown twice:

1. the **original 512 × 512** angiographic frame, and
2. the corresponding **224 × 224** image that is actually passed to the model.

The side-by-side view makes it possible to judge whether vessel detail is lost during
resizing, and whether the model's errors deserve closer clinical inspection.

> The notebook reads the patient-level predictions and checkpoint produced by
> `cadica-train`. Run `cadica-train --fold 1` first if they are missing.


In [ ]:
from pathlib import Path
import sys

# Locate the repository even when Jupyter starts in the notebooks/ directory.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "src" / "cadica").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root (src/cadica).")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms

from cadica.config import (
    CHECKPOINTS_DIR,
    CLASS_NAMES,
    RESULTS_DIR,
    val_fold_path,
)
from cadica.data.dataset import default_transform, resolve_image_path
from cadica.models import CADICACNN

FOLD = 1
MODEL_SIZE = (224, 224)
N_FRAMES_PER_PATIENT = 4

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.titlesize": 10,
        "figure.titlesize": 12,
    }
)
pd.set_option("display.max_colwidth", 60)

## Load predictions, validation manifest, and checkpoint

In [ ]:
prediction_path = RESULTS_DIR / f"patient_predictions_fold_{FOLD}.csv"
checkpoint_path = CHECKPOINTS_DIR / f"baseline_fold_{FOLD}.pt"

missing = [path for path in (prediction_path, checkpoint_path) if not path.exists()]
if missing:
    formatted = "\n".join(f"  - {path}" for path in missing)
    raise FileNotFoundError(
        "Missing training outputs for error analysis:\n"
        f"{formatted}\n"
        f"Run `cadica-train --fold {FOLD}` first."
    )

predictions = pd.read_csv(prediction_path)
val_df = pd.read_csv(val_fold_path(FOLD))

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CADICACNN(num_classes=len(CLASS_NAMES))
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device).eval()

print(f"Checkpoint: {checkpoint_path}")
print(f"Checkpoint metrics: {checkpoint['final_metrics']}")
print(f"Device: {device}")
print(
    f"Validation frames: {len(val_df):,} "
    f"from {val_df['patient_id'].nunique()} patients"
)

## Patient-level summary

Accuracy here is **patient-level**: frame probabilities are averaged within each
patient before taking the argmax, matching the training evaluation.


In [ ]:
probability_columns = [f"p_{name}" for name in CLASS_NAMES]
class_name_by_index = dict(enumerate(CLASS_NAMES))

summary = predictions.copy()
summary["actual_name"] = summary["actual"].map(class_name_by_index)
summary["predicted_name"] = summary["predicted"].map(class_name_by_index)
summary["confidence"] = summary[probability_columns].max(axis=1)
summary["correct"] = summary["actual"] == summary["predicted"]

val_patients = set(val_df["patient_id"].unique())
prediction_patients = set(summary["patient_id"])
if val_patients != prediction_patients:
    raise ValueError(
        "Validation patients and prediction patients do not match: "
        f"missing={sorted(val_patients - prediction_patients)}, "
        f"extra={sorted(prediction_patients - val_patients)}"
    )

summary = summary.sort_values(
    ["correct", "patient_id"], ascending=[False, True]
).reset_index(drop=True)

display(
    summary[
        ["patient_id", "actual_name", "predicted_name", "confidence", "correct"]
    ].rename(
        columns={
            "patient_id": "patient",
            "actual_name": "actual",
            "predicted_name": "predicted",
            "confidence": "confidence",
        }
    )
)

n_correct = int(summary["correct"].sum())
n_total = len(summary)
print(f"Correctly classified: {n_correct} / {n_total}")
print(f"Incorrectly classified: {n_total - n_correct} / {n_total}")

## Frame selection and plotting helpers

The displayed frames are evenly spaced across each patient's validation frames, so
they usually cover several angiographic videos. The 224 × 224 panel uses the same
`torchvision` resize operation as `CADICADataset`, not a separate PIL resize.

Each resized panel is annotated with the model's **frame-level** prediction and
confidence. This helps separate a patient-level error caused by a few misleading
frames from one where most frames agree with the wrong class.


In [ ]:
resize_for_model = transforms.Resize(MODEL_SIZE)
preprocess_for_model = default_transform()


def select_patient_frames(patient_id, n_frames=N_FRAMES_PER_PATIENT):
    """Pick evenly spaced validation frames across all videos for one patient."""
    patient_rows = val_df.loc[val_df["patient_id"] == patient_id].reset_index(
        drop=True
    )
    if patient_rows.empty:
        raise ValueError(f"No validation frames found for {patient_id!r}.")

    n = min(n_frames, len(patient_rows))
    positions = np.linspace(0, len(patient_rows) - 1, n).round().astype(int)
    positions = np.unique(positions)
    return patient_rows.iloc[positions].reset_index(drop=True)


def load_original_and_resized(image_path):
    """Return the full-resolution RGB frame and its 224 × 224 model input."""
    with Image.open(resolve_image_path(image_path)) as image:
        original = image.convert("RGB")
    resized = resize_for_model(original)
    return original, resized


def predict_frames(image_paths):
    """Run the checkpoint on frames using the dataset's deterministic transform."""
    tensors = []
    for image_path in image_paths:
        with Image.open(resolve_image_path(image_path)) as image:
            tensors.append(preprocess_for_model(image.convert("RGB")))

    with torch.inference_mode():
        logits = model(torch.stack(tensors).to(device))
        probabilities = torch.softmax(logits, dim=1)

    return probabilities.cpu().numpy()


def plot_patient(patient_id):
    """Show sampled originals and model inputs for one patient."""
    patient_summary = summary.loc[summary["patient_id"] == patient_id].iloc[0]
    frames = select_patient_frames(patient_id)

    originals = []
    resized_images = []
    for image_path in frames["image_path"]:
        original, resized = load_original_and_resized(image_path)
        originals.append(original)
        resized_images.append(resized)

    frame_probabilities = predict_frames(frames["image_path"].tolist())

    fig, axes = plt.subplots(
        len(frames),
        2,
        figsize=(7.2, 3.8 * len(frames)),
        squeeze=False,
    )

    for row_index in range(len(frames)):
        original = originals[row_index]
        model_input = resized_images[row_index]
        probabilities = frame_probabilities[row_index]

        original_width, original_height = original.size
        frame_class = CLASS_NAMES[int(probabilities.argmax())]
        frame_confidence = probabilities.max()

        axes[row_index, 0].imshow(original)
        axes[row_index, 0].set_title(
            f"Original {original_width} × {original_height}"
        )

        axes[row_index, 1].imshow(model_input)
        axes[row_index, 1].set_title(
            f"Model input {MODEL_SIZE[0]} × {MODEL_SIZE[1]} | "
            f"frame pred: {frame_class} ({frame_confidence:.2f})"
        )

        for axis in axes[row_index]:
            axis.axis("off")

    verdict = "correct" if patient_summary["correct"] else "incorrect"
    fig.suptitle(
        f"{patient_id} — actual: {patient_summary['actual_name']} | "
        f"predicted: {patient_summary['predicted_name']} ({verdict}) | "
        f"patient confidence: {patient_summary['confidence']:.3f}",
        y=1.0,
    )
    fig.tight_layout()
    plt.show()

## Correctly classified patients

Inspect whether the vessel tree stays legible after downsampling. If the model is
correctly classifying these patients, the resized panels should still contain the
anatomy that distinguishes the severity classes.


In [ ]:
correct_patients = summary.loc[summary["correct"], "patient_id"].tolist()
print("Correctly classified patients:", ", ".join(correct_patients))

for patient_id in correct_patients:
    plot_patient(patient_id)

## Incorrectly classified patients

Compare these panels with the correct examples. Look for:

- vessels that disappear or become ambiguous at 224 × 224,
- frames whose individual predictions disagree with the patient-level prediction,
- low-contrast or very small coronary segments that may be clinically important.

These are candidates for closer clinical inspection and for later explainability
work; they are not yet evidence that the architecture is the limiting factor.


In [ ]:
incorrect_patients = summary.loc[
    ~summary["correct"], "patient_id"
].tolist()
print("Incorrectly classified patients:", ", ".join(incorrect_patients))

for patient_id in incorrect_patients:
    plot_patient(patient_id)